# EQ2425 Project 2: Visual search system

In [ ]:
import cv2 as cv
import numpy as np
import os
import matplotlib.pyplot as plt

## Section 2: Feature extraction

In [ ]:
folder_path = os.path.join("Data2", "server")

folder_path_client = os.path.join("Data2", "client")

def extracting_features(folderpath):

    image_list = []
    object_numbers = []          # object number of every image, read from the file name

    for filename in os.listdir(folderpath):
        path = os.path.join(folderpath, filename)

        image = cv.imread(path, cv.IMREAD_GRAYSCALE)

        if image is not None:
            image_list.append(image)
            object_numbers.append(int(filename.split("_")[0][3:]))   # "obj13_2.JPG" -> 13

    print("Number of images in the folder ", len(image_list))

    Sift = cv.SIFT_create(nfeatures = 3000)

    descriptors_list = []
    for image in image_list:

        keypoints, descriptors = Sift.detectAndCompute(image, None)

        descriptors_list.append(descriptors)

    return descriptors_list, object_numbers


def dictionary(descriptors_list, object_numbers):

    # key = object number, value = all descriptors of that object
    # works for objects with 2, 3 or 4 images (obj26, obj37, obj38)
    descriptor_dictionary = {}

    for descriptors, obj in zip(descriptors_list, object_numbers):

        if obj in descriptor_dictionary:
            descriptor_dictionary[obj] = np.concatenate((descriptor_dictionary[obj], descriptors))
        else:
            descriptor_dictionary[obj] = descriptors

    return descriptor_dictionary

In [ ]:
server_image_descriptors, server_object_numbers = extracting_features(folder_path)

server_image_dictionary = dictionary(server_image_descriptors, server_object_numbers)

client_image_descriptors, client_object_numbers = extracting_features(folder_path_client)

client_image_dictionary = dictionary(client_image_descriptors, client_object_numbers)

In [ ]:
# Check the grouping and answer 2(a), 2(b)
print("Objects in server / client:", len(server_image_dictionary), len(client_image_dictionary))
print("obj26 (2 images):", len(server_image_dictionary[26]), "  obj37 (4 images):", len(server_image_dictionary[37]))

print("2(a) average features per database object:", np.mean([len(d) for d in server_image_dictionary.values()]))
print("2(b) average features per query object:   ", np.mean([len(d) for d in client_image_dictionary.values()]))

## Section 3: Vocabulary tree

In [ ]:
# ============================================================
# STEP A - Prepare descriptors for vocabulary tree
# ============================================================

all_descriptors = []
object_ids = []

# server_image_dictionary:
# key   = object number
# value = all SIFT descriptors of that object
for object_id, descriptors in server_image_dictionary.items():

    # Add all descriptors of this object
    all_descriptors.append(descriptors)

    # For every descriptor, store the corresponding object ID
    object_ids.extend([object_id] * len(descriptors))


# Convert everything to NumPy arrays
all_descriptors = np.vstack(all_descriptors).astype(np.float32)
object_ids = np.array(object_ids, dtype=np.int32)


print("Shape of all descriptors:", all_descriptors.shape)
print("Shape of object IDs:", object_ids.shape)
print("Number of descriptors:", len(all_descriptors))
print("Number of object IDs:", len(object_ids))

# Check that the two arrays correspond
print("Same number of descriptors and object IDs:",
      len(all_descriptors) == len(object_ids))

In [ ]:
# ============================================================
# STEP B - Hierarchical K-Means Vocabulary Tree
# ============================================================

def hi_kmeans(data, b, depth):

    # Indices of the descriptors in the original dataset
    indices = np.arange(len(data))

    # NEW: list that collects every leaf, its position = leaf_id
    leaves = []

    # Start the recursive construction from the root
    tree = build_tree(data, indices, b, depth, 0, leaves)

    # NEW: keep the list of leaves inside the tree
    tree["leaves"] = leaves

    return tree


def build_tree(data, indices, b, depth, current_depth, leaves):

    # --------------------------------------------------------
    # LEAF: requested depth reached, or too few descriptors
    # --------------------------------------------------------
    if current_depth == depth or len(data) < b:

        leaf = {
            "is_leaf": True,
            "depth": current_depth,
            "indices": indices,
            "leaf_id": len(leaves)          # NEW: number of this visual word
        }
        leaves.append(leaf)                 # NEW
        return leaf


    # SIFT descriptors must be float32 for OpenCV kmeans
    data_float = data.astype(np.float32)


    # --------------------------------------------------------
    # K-MEANS
    # --------------------------------------------------------
    criteria = (
        cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER,
        20,                                 # CHANGED from 100
        1.0
    )

    # --------------------------------------------------------
    # APPLY K-MEANS
    # --------------------------------------------------------
    compactness, labels, centers = cv.kmeans(
        data_float,
        b,
        None,
        criteria,
        1,                                  # CHANGED from 10
        cv.KMEANS_PP_CENTERS
    )

    labels = labels.ravel()


    # --------------------------------------------------------
    # CREATE CURRENT NODE
    # --------------------------------------------------------
    node = {
        "is_leaf": False,
        "depth": current_depth,
        "centroids": centers,               # 3(a): the b centers stored in the node
        "children": []
    }


    # --------------------------------------------------------
    # RECURSIVELY BUILD THE b CHILDREN
    # --------------------------------------------------------
    for cluster_id in range(b):

        mask = (labels == cluster_id)
        child_data = data[mask]
        child_indices = indices[mask]

        child = build_tree(
            child_data,
            child_indices,
            b,
            depth,
            current_depth + 1,
            leaves                          # NEW: pass the list down
        )

        node["children"].append(child)

    return node

In [ ]:
# ============================================================
# STEP B2 - Number the leaves (visual words) and check the tree
# ============================================================

def get_leaves(node, leaves):

    if node["is_leaf"]:
        node["leaf_id"] = len(leaves)      # give every leaf a number 0, 1, 2 ...
        leaves.append(node)
    else:
        for child in node["children"]:
            get_leaves(child, leaves)

    return leaves


# small tree first, easy to look at
tree = hi_kmeans(all_descriptors, 4, 3)
leaves = get_leaves(tree, [])

print("Number of leaves:", len(leaves), " (4^3 =", 4**3, ")")
print("Descriptors in all leaves:", sum(len(leaf["indices"]) for leaf in leaves))
print("Descriptors in total:     ", len(all_descriptors))

### Look at the tree

Each dot is a node. The bigger the dot, the more descriptors it holds. The root at the top holds all of them, k-means splits them into b groups per level, and the bottom row are the leaves (visual words).

In [ ]:
def draw_tree(node, ax, counter, level=0):
    # draws the children first, then puts the parent above the middle of its children
    # returns the x position and the number of descriptors of this node

    if node["is_leaf"]:
        x = counter[0]                     # next free position on the bottom row
        counter[0] += 1
        n = len(node["indices"])
    else:
        child_x = []
        n = 0
        for child in node["children"]:
            cx, cn = draw_tree(child, ax, counter, level + 1)
            child_x.append(cx)
            n += cn
        x = np.mean(child_x)
        for cx in child_x:
            ax.plot([x, cx], [-level, -(level + 1)], color="gray", linewidth=0.8, zorder=1)

    ax.scatter(x, -level, s=20 + 400 * n / len(all_descriptors) * (level + 1), color="C0", zorder=2)
    if level <= 1:
        ax.text(x, -level + 0.15, f"{n:,}", ha="center", fontsize=9)
    return x, n


fig, ax = plt.subplots(figsize=(14, 5))
draw_tree(tree, ax, [0])
ax.set_yticks([0, -1, -2, -3], ["root", "level 1", "level 2", "level 3 (leaves)"])
ax.set_xticks([])
ax.set_title("Vocabulary tree b=4, depth=3: numbers = descriptors in the node")
plt.show()

In [ ]:
# How many descriptors ended up in each leaf (visual word)
leaf_sizes = [len(leaf["indices"]) for leaf in leaves]

plt.figure(figsize=(14, 3.5))
plt.bar(range(len(leaf_sizes)), leaf_sizes)
plt.xlabel("leaf number (visual word)")
plt.ylabel("number of descriptors")
plt.title("Descriptors per leaf, b=4, depth=3")
plt.show()

In [ ]:
# The three trees the task asks for (the big one takes about a minute)
trees = {}
for b, depth in [(4, 3), (4, 5), (5, 7)]:
    t = hi_kmeans(all_descriptors, b, depth)
    l = get_leaves(t, [])
    trees[(b, depth)] = (t, l)
    print(f"b={b} depth={depth}: {len(l):>6} leaves (max {b**depth:>6}),",
          "all descriptors in leaves:", sum(len(leaf["indices"]) for leaf in l) == len(all_descriptors))

In [ ]:
def add_tfidf_info(tree, object_ids, num_objects):

    for leaf in tree["leaves"]:

        # Object IDs of all descriptors that reached this leaf
        leaf_object_ids = object_ids[leaf["indices"]]

        # Find how many descriptors from each object are in this leaf
        unique_objects, counts = np.unique(
            leaf_object_ids,
            return_counts=True
        )

        # TF: number of occurrences for each object
        tf = {}

        for object_id, count in zip(unique_objects, counts):
            tf[int(object_id)] = int(count)

        # DF: number of different objects containing this visual word
        df = len(unique_objects)

        # IDF
        idf = np.log(1+ num_objects / df)

        # Store information in the leaf
        leaf["tf"] = tf
        leaf["df"] = df
        leaf["idf"] = idf

In [ ]:
add_tfidf_info(tree, object_ids, len(server_image_dictionary))

for leaf in tree["leaves"]:
    print(leaf["leaf_id"], leaf["idf"])

## Section 4: Querying

Next: STEP C (leaf counts and idf), then query and recall.